# Movie Script Analysis
In this notebook, we'll analyze a movie script to provide contextual information to feed our conversational agent with advanced search capability. We'll use preprocess the original script, then extract the content from the script using BDA, and ingest the content into Bedrock Knowledge Bases to power semantic search. 

In [ ]:
# Make sure you download the latest botocore and boto3 libraries.
import shutil
import subprocess
import sys

def ensure_uv_installed():
    if shutil.which("uv") is None:
        print("🔧 'uv' not found. Installing with pip...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"])
    else:
        print("✅ 'uv' is already installed.")

def uv_install(*packages):
    ensure_uv_installed()
    uv_path = shutil.which("uv")
    print(f"📦 Installing {', '.join(packages)} using uv...")
    subprocess.check_call([uv_path, "pip", "install", *packages])

uv_install("botocore", "boto3")

In [ ]:
import os
os.environ["AWS_PROFILE"] = "your AWS PROFILE"
os.environ["AWS_DEFAULT_REGION"] = "your aws region"

In [ ]:
import boto3
from datetime import datetime
import time
import uuid

bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
region = boto3.Session().region_name
s3_client = boto3.client("s3")
 

In [ ]:
data_bucket = "enter your bucket name here" # You can find the bucket name in the CDK output with the output name: BedrockKBStack.KnowledgeBaseDataBucket
data_prefix = "documents"
bda_prefix = "bda"
output_prefix = "output"

print('data S3 bucket:', data_bucket)

In [ ]:
response = bda_client.create_data_automation_project(
    projectName=f'bda-project-{str(uuid.uuid4())[0:4]}',
    projectDescription='BDA Conversation Agent script analysis project',
    projectStage='DEVELOPMENT',
    standardOutputConfiguration={
        'document': {
            'extraction': {
                'granularity': {
                    'types': ['DOCUMENT', 'PAGE', 'ELEMENT']
                },
                'boundingBox': {
                    'state': 'DISABLED'
                }
            },
            'generativeField': {
                'state': 'DISABLED'
            },
            'outputFormat': {
                'textFormat': {
                    'types': ['PLAIN_TEXT', 'MARKDOWN']
                },
                'additionalFileFormat': {
                    'state': 'DISABLED'
                }
            }
        }
    }
)

In [ ]:
bda_project_arn = response["projectArn"]
print("BDA project ARN:", bda_project_arn)

# Download Original Script 
In the following step, we are going to upload the movie script for "Tears Of Steel", which is available in this [URL](https://studio.blender.org/projects/tears-of-steel/55f344892beb3300251b0172/?asset=34). Please note the script, and other assets are available to users who are registered with Blender Studio. Please follow the instructions on the website on how to register an account. 

Once the registration is complete, download the script into a local directory, then continue to steps as followed.

In [ ]:
sample_script = "the local filepath of the script" # The movie script location downloaded locally
sample_script_basename = os.path.basename(sample_script)
bda_s3_input_location = f"s3://{data_bucket}/{bda_prefix}/{sample_script_basename}"

upload the script to S3 bucket location

In [ ]:
!aws s3 cp $sample_script $bda_s3_input_location

Triggers a BDA job to process the movie script

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': bda_s3_input_location
    },
    outputConfiguration={
        's3Uri': f's3://{data_bucket}/{output_prefix}'
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': bda_project_arn,
        'stage': 'DEVELOPMENT'
    },
    notificationConfiguration={
        'eventBridgeConfiguration': {
            'eventBridgeEnabled': False
        }
    },
    dataAutomationProfileArn=f'arn:aws:bedrock:{region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

In [ ]:
invocation_arn = response.get("invocationArn")
print("BDA task started:", invocation_arn)

Wait for the BDA job to complete

In [ ]:
status, status_response = None, None
while status not in ["Success","ServiceError","ClientError"]:
    status_response = bda_runtime_client.get_data_automation_status(
        invocationArn=invocation_arn
    )
    status = status_response.get("status")
    print(f"{datetime.now().strftime('%H:%M:%S')} : BDA task: {status}")
    time.sleep(5)

output_config = status_response.get("outputConfiguration",{}).get("s3Uri")
print("Output configureation file:", output_config)

In [ ]:
import utils 
from pprint import pprint
import json
from pathlib import Path

config_data = utils.read_json_on_s3(output_config,s3_client)
pprint(config_data)

Extract the BDA output

In [ ]:
Path("temp-outputs/bda").mkdir(parents=True, exist_ok=True)

result_uri = config_data["output_metadata"][0]["segment_metadata"][0]["standard_output_path"]
result_data = utils.read_json_on_s3(result_uri,s3_client)

# Because the file is large, we are going to save this file so we could view the content using the built-in JSON file explorer in your IDE or jupyterlab.
with open("temp-outputs/bda/document_output.json", "w") as f:
    f.write(json.dumps(result_data))

In [ ]:
pprint(result_data["document"]["representation"]["text"])

Clean the script from the extracted output before ingesting into Bedrock Knowledge Bases for Semantic Search. 

In [ ]:
def remove_lines(text):
    lines = text.split('\n')
    return '\n'.join(line for line in lines if not (len(line) == 0 or line.startswith('CONTINUED:') or line.startswith('(CONTINUED)')))

In [ ]:
cleaned_text= remove_lines(result_data["document"]["representation"]["text"])
pprint(cleaned_text)

In [ ]:
import os
cleaned_filename = f'{".".join(os.path.basename(bda_s3_input_location).split(".")[:-1])}_cleaned.txt'
cleaned_filename = cleaned_filename.replace(" ","")
with open(f"temp-outputs/{cleaned_filename}", "w") as f:
    f.write(remove_lines(result_data["document"]["representation"]["text"]))

Uploads the cleaned extracted script to S3 bucket for data ingestion into Knowledge Bases for Bedrock.

In [ ]:
!aws s3 cp temp-outputs/$cleaned_filename s3://$data_bucket/$data_prefix/

In addition to the script content, we are also supplementing a metadata file for the script to allow Knowledge Bases for Bedrock to perform hybrid search. This would allow the semantic search with filter with the title of the movie. 

In [ ]:
video_name = "tears-of-steel.mp4" # or replace with the movie file corresponds to this particular script.
document_script_vector_metadata = {"metadataAttributes": {"video_name" : video_name}}
document_script_vector_metadata_json_filename = f"{cleaned_filename}.metadata.json"
with open(f"temp-outputs/{document_script_vector_metadata_json_filename}", "w") as f:
    f.write(json.dumps(document_script_vector_metadata))


Uploads the metadata file to S3 bucket location

In [ ]:
!aws s3 cp temp-outputs/$document_script_vector_metadata_json_filename s3://$data_bucket/$data_prefix/

## Bedrock Knowledge Bases Data Ingestion
In the following section, we are going to initiate a data synchronization process for Knowledge Bases for Bedrock to ingest the script content and metadata into the vector database. Once the process is complete, we'll be able to use the Knowledge Bases for semantic search. 

In [ ]:
bedrock_agent_client = boto3.client('bedrock-agent')

knowledge_base_id = "your Knowledge Bases ID" # provide the Bedrock Knowledge Bases ID. You can find this value from the CDK output : BedrockKBStack.KnowledgeBaseId
data_source_id = "Your Data Source ID" # provide the Data Source ID for this data ingestion job. You can find this value from the CDK output: BedrockKBStack.KnowledgeBaseDataSourceId


In [ ]:
response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=knowledge_base_id,
    dataSourceId=data_source_id
)

In [ ]:
response